## 과제 3: 삼성전자 일별 시세 정보 가기

### 개요
이 과제에서는 네이버 페이 증권 웹사이트에서 크롬 개발자 도구를 활용하여 주식 시세 데이터를 제공하는 API를 직접 찾아내고 분석합니다. 
네트워크 요청 분석을 통해 공개되지 않은 API 엔드포인트를 찾아내고, 이를 활용하여 체계적으로 데이터를 수집하는 과정을 실습합니다. 


### 목표

삼성전자일별 시세만을 보여주는 URL 을 크롬개발자 도구를 활용해 찾아낸 후, 해당 URL의 정보를 파이썬으로 가져온다. 

- 일자별 데이터 파싱은 하지 않고 **일자별 시세에 해당하는 URL**의 정보만 가져올 수 있으면 됩니다. (HTML 정보가 출력됨)

[네이버 증권 삼성전자](https://finance.naver.com/item/sise.naver?code=005930데이터)


---

### 1. 크롬 개발자도구를 통한 데이터 수집

![엔드포인트찾아내기](../../../images/screenshot%202025-10-28%20오후%203.30.26.png)

- 서버 주소: https://finance.naver.com/item/sise_day.naver?code=005930&page=1

- 파라미터: code, page

- 요청방식: GET

---

### 2. 1페이지 HTML 가져오기(파싱없이)

In [9]:
import requests

BASE = "https://finance.naver.com/item/sise_day.naver"
params = {"code": "005930", "page": 1}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

resp = requests.get(BASE, params=params, headers=headers, timeout=10)

# 네이버 일별시세 페이지는 EUC-KR(=cp949) 인코딩
resp.encoding = "cp949"  # 또는 "euc-kr"

print("요청 URL:", resp.url)
print(resp.text[:1200])  # HTML 앞부분만 맛보기로 출력


요청 URL: https://finance.naver.com/item/sise_day.naver?code=005930&page=1

<html lang="ko">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=euc-kr">
<title>네이버페이 증권</title>

<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/newstock.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/common.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/finance_header.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/layout.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/main.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock/static.pc/20251023172802/css/newstock2.css">
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/imgstock

---

### 2. 1-10 페이지 HTML 가져오기(파싱없이)

In [10]:
import requests
import time

BASE = "https://finance.naver.com/item/sise_day.naver"

def fetch_page_html(code, page):
    """네이버 일별시세 HTML 가져오기"""
    params = {"code": code, "page": page}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    r = requests.get(BASE, params=params, headers=headers, timeout=10)
    r.raise_for_status()
    r.encoding = "cp949"  # euc-kr 인코딩
    return r.text

# 삼성전자 1~3페이지 HTML 저장
for page in range(1, 4):
    html = fetch_page_html("005930", page)
    with open(f"sise_day_005930_p{page}.html", "w", encoding="cp949", errors="ignore") as f:
        f.write(html)
    print(f"{page}페이지 저장 완료")
    time.sleep(0.5)


1페이지 저장 완료
2페이지 저장 완료
3페이지 저장 완료


---

### 3. 표 파싱

HTML 파일 안에서 \<table class="type2"> 표를 찾아   
그 안의 <tr>(행)들을 돌면서,    
날짜 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량 뽑아내기

In [11]:
# pip install bs4

In [12]:
# BeautifulSoup: HTML 문서를 구조적으로 다룰 수 있게 해주는 라이브러리
# 문자열 형태의 HTML을 DOM(트리 구조) 로 변환해서 태그 단위로 검색할 수 있게 만들어줌
from bs4 import BeautifulSoup

def parse_table(html):
    """일별시세 표만 파싱"""
    soup = BeautifulSoup(html, "html.parser")
    # select_one(): CSS 선택자(selector) 문법을 사용해서 특정 태그를 1개 찾아줌
    table = soup.select_one("table.type2")

    rows = []
    # table 안의 모든 <tr>(행)을 리스트로 가져옴
    for tr in table.select("tr"):
        tds = tr.select("td")
        if len(tds) != 7:
            continue
        cols = [td.get_text(strip=True) for td in tds]
        # date: 날짜, close: 종가, diff: 전일비, open_: 시가, high: 고가, low: 저가, volume: 거래량
        date, close, diff, open_, high, low, volume = cols
        if not date or date == "날짜":
            continue
        rows.append([date, close, diff, open_, high, low, volume])

    return rows

# 테스트
with open("sise_day_005930_p1.html", encoding="cp949") as f:
    html = f.read()

parsed = parse_table(html)
for row in parsed[:5]:
    print(row)


['2025.10.28', '99,500', '하락2,500', '100,900', '101,000', '99,100', '19,830,309']
['2025.10.27', '102,000', '상승3,200', '101,300', '102,000', '100,600', '22,169,970']
['2025.10.24', '98,800', '상승2,300', '97,900', '99,000', '97,700', '18,801,925']
['2025.10.23', '96,500', '하락2,100', '96,800', '98,500', '96,300', '18,488,581']
['2025.10.22', '98,600', '상승1,100', '97,100', '98,600', '95,500', '15,937,611']


---

### 4. pandas로 DataFrame 정리 및 CSV 저장

In [13]:
import pandas as pd

def to_dataframe(parsed_rows):
    """파싱 결과를 DataFrame으로 변환"""
    df = pd.DataFrame(parsed_rows, columns=["날짜", "종가", "전일비", "시가", "고가", "저가", "거래량"])

    # 문자열 → 숫자/날짜 변환
    df["날짜"] = pd.to_datetime(df["날짜"], format="%Y.%m.%d")
    for col in ["종가", "전일비", "시가", "고가", "저가", "거래량"]:
        df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce")

    return df.sort_values("날짜", ascending=False).reset_index(drop=True)

# DataFrame 만들기
df = to_dataframe(parsed)
print(df.head())

# CSV로 저장
df.to_csv("samsung_day_005930.csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료")


          날짜      종가  전일비      시가      고가      저가       거래량
0 2025-10-28   99500  NaN  100900  101000   99100  19830309
1 2025-10-27  102000  NaN  101300  102000  100600  22169970
2 2025-10-24   98800  NaN   97900   99000   97700  18801925
3 2025-10-23   96500  NaN   96800   98500   96300  18488581
4 2025-10-22   98600  NaN   97100   98600   95500  15937611
CSV 저장 완료
